In [ ]:
#INSTALACIÓN (ejecuta una sola vez)
#!pip install requests beautifulsoup4 pandas gspread gspread_dataframe --quiet no 
import os
import json
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

import gspread
from gspread_dataframe import set_with_dataframe
from gspread.exceptions import SpreadsheetNotFound, WorksheetNotFound
from oauth2client.service_account import ServiceAccountCredentials

In [ ]:
#Autentication bot
# Leer la llave secreta GitHub
credenciales_json = json.loads(os.environ['GOOGLE_CREDENTIALS'])

# Definir los permisos (Drive y Sheets)
scope = [
    "https://spreadsheets.google.com/feeds", 
    "https://www.googleapis.com/auth/drive"
]

creds = ServiceAccountCredentials.from_json_keyfile_dict(credenciales_json, scope)
gc = gspread.authorize(creds)


In [ ]:
#Aqui el calendario F1 2026
headers = {'User-Agent': 'Mozilla/5.0'}

url = "https://www.marca.com/motor/formula1/calendario.html"
response = requests.get(url, headers=headers)
response.encoding = 'ISO-8859-1'

soup = BeautifulSoup(response.text, 'html.parser')
text_content = soup.get_text(separator='\n', strip=True)

# Dividimos en bloques
blocks = re.split(r'(GP [^:\n]{2,40}:)', text_content)

gps = []
current_gp = None

for part in blocks:
    part = part.strip()
    # Si empieza por GP y termina en dos puntos (sin importar qué haya en medio)
    if re.match(r'GP [^:\n]{2,40}:', part):
        if current_gp:
            gps.append(current_gp)
        current_gp = {'GP': part.strip(':')}
    elif current_gp is not None:
        current_gp['raw'] = current_gp.get('raw', '') + '\n' + part

if current_gp:
    gps.append(current_gp)

# ==================== PARSEO BASE ====================
data = []
for gp in gps:
    raw = gp.get('raw', '')
    row = {'GP': gp['GP']}

    # Fechas, Circuito, Longitud, Curvas, Record (IGUAL QUE TU BASE)
    fechas_match = re.search(r'(\d{1,2}-\d{1,2}(?: de)? [A-Za-z]+(?: \d{4})?)', raw, re.I)
    row['Fechas'] = fechas_match.group(1).strip() if fechas_match else 'N/A'

    circuito_match = re.search(r'CIRCUITO:?\s*([^\n]+)', raw, re.I) or re.search(r'(Albert Park|Shanghai|Suzuka|Hard Rock|Gilles Villeneuve|Montecarlo)[^\n]*', raw, re.I)
    row['Circuito'] = circuito_match.group(1).strip() if circuito_match else 'N/A'

    long_match = re.search(r'LONGITUD:?\s*(\d+)\s*metros', raw, re.I)
    row['Longitud (m)'] = long_match.group(1) if long_match else 'N/A'

    curvas_match = re.search(r'CURVAS:?\s*(\d+)\s*\((\d+)\s*izda\s*[\|/]\s*(\d+)\s*dcha', raw, re.I)
    row['Curvas'] = f"{curvas_match.group(1)} ({curvas_match.group(2)} izda / {curvas_match.group(3)} dcha)" if curvas_match else 'N/A'


# ==================== RÉCORD DE VUELTA (SÚPER DIRECTO Y FLEXIBLE) ====================
    # Aplanamos los saltos de línea
    raw_flat = raw.replace('\n', ' ')

    # (.*?): Atrapa el nombre del piloto
    # \b(\d{1,2}:[\d\.]+): Atrapa el tiempo exacto
    rec_match = re.search(r'R.?CORD:?\s*(.*?)\s*[-–—]?\s*\b(\d{1,2}:[\d\.]+)', raw_flat, re.I)

    if rec_match:
        piloto = rec_match.group(1).strip().rstrip('-').strip()

        # --- LIMPIEZA INTELIGENTE (SIN ELIMINAR LETRAS) ---
        # 1. Quitamos espacios invisibles (html \xa0)
        piloto = piloto.replace('\xa0', ' ')

        # 2. Reparamos los errores de la web "cambiando" en vez de "borrando"
        # Si la web manda PÃ©rez, Prez, o cualquier variante rota, la arreglamos:
        piloto = piloto.replace('PÃ©rez', 'Pérez').replace('Prez', 'Pérez')

        # (Opcional pero infalible) Si la web lo destrozó mucho, buscamos la P y rez:
        piloto = re.sub(r'P[^a-zA-Z\s]*rez', 'Pérez', piloto, flags=re.I)

        # 3. Limpiamos espacios dobles
        piloto = re.sub(r'\s+', ' ', piloto).strip()
        # --------------------------------------------------

        tiempo = rec_match.group(2).strip()

        # Buscamos el año en el trocito de texto justo después del tiempo
        texto_post_tiempo = raw_flat[rec_match.end():rec_match.end()+25]
        anio_match = re.search(r'(\d{4})', texto_post_tiempo)

        if anio_match:
            row['Record vuelta'] = f"{piloto} - {tiempo} ({anio_match.group(1)})"
        else:
            row['Record vuelta'] = f"{piloto} - {tiempo}"
    else:
        row['Record vuelta'] = 'N/A'
    # =====================================================================

    ganador_match = re.search(r'(?:GANADOR|Ganador)\s*(?:2025|anterior)?[:*\s]*([A-Za-záéíóúñ ]{4,})(?=\s*(?:Pole|POLE|Ir arriba|$))', raw, re.I | re.DOTALL)
    row['Ganador anterior'] = ganador_match.group(1).strip() if ganador_match else 'N/A'

    pole_match = re.search(r'(?:POLE|Pole)\s*(?:2025|anterior)?[:*\s]*([A-Za-záéíóúñ ]{4,})(?=\s*(?:Viernes|Horarios|Ir arriba|$))', raw, re.I | re.DOTALL)
    row['Pole anterior'] = pole_match.group(1).strip() if pole_match else 'N/A'

    podios = re.findall(r'(?:PRIMERO|SEGUNDO|TERCERO)\s*([A-Za-záéíóúñ ]+)', raw, re.I)
    if len(podios) >= 3:
        row['Podio reciente'] = f"1º {podios[0].strip()} | 2º {podios[1].strip()} | 3º {podios[2].strip()}"
    else:
        podios2 = re.findall(r'\*(?:PRIMERO|SEGUNDO|TERCERO)([A-Za-záéíóúñ ]+)', raw, re.I)
        if len(podios2) >= 3:
            row['Podio reciente'] = f"1º {podios2[0].strip()} | 2º {podios2[1].strip()} | 3º {podios2[2].strip()}"
        else:
            row['Podio reciente'] = 'N/A'


# ==================== HORARIOS COMPLETOS (BLOQUEO DE FECHAS EXTRA) ====================
    # TRUCO 1: Quitamos los saltos de línea (\n) SOLO para buscar los horarios
    raw_horarios = raw.replace('\n', ' ')

    patron_dias = r'(JUEVES|VIERNES|SÁBADO|bado|DOMINGO)\s+(\d{1,2})'
    iter_dias = list(re.finditer(patron_dias, raw_horarios, re.I))

    correcciones_dias = {'bado': 'Sábado', 'jueves': 'Jueves', 'viernes': 'Viernes', 'sábado': 'Sábado', 'domingo': 'Domingo'}
    horarios_completos = []

    # TRUCO 2: Memoria de días. Si un día se repite, el GP ya terminó.
    dias_procesados = set()

    for i in range(len(iter_dias)):
        match_actual = iter_dias[i]
        dia_raw = match_actual.group(1).lower()
        num_dia = match_actual.group(2)

        nombre_dia = correcciones_dias.get(dia_raw, dia_raw.capitalize())

        # --- EL CORTAFUEGOS ---
        # Si ya leímos un Viernes/Sábado, etc., y aparece otro, detenemos la lectura
        if nombre_dia in dias_procesados:
            break
        dias_procesados.add(nombre_dia)
        # ----------------------

        inicio_texto = match_actual.end()
        fin_texto = iter_dias[i+1].start() if i + 1 < len(iter_dias) else len(raw_horarios)
        texto_dia = raw_horarios[inicio_texto:fin_texto]

        sesiones = re.findall(r'([A-Za-záéíóúñÁÉÍÓÚÑ\s\d]{1,40}):\s*(\d{2}:\d{2})', texto_dia)

        clean_s = []
        for s_nom, s_hora in sesiones:
            nom = s_nom.strip()
            nom = re.sub(r'^[\d:\s]+', '', nom).strip()
            nom_l = nom.lower()

            # Rescate de Clasificación general
            if len(nom) < 3 or ("carrera" in nom_l and nom_l != "carrera" and "sprint" not in nom_l):
                if "libres" not in nom_l:
                    nom = "Clasificación"

            nom = nom.capitalize()

            # Limpieza estética final de la web de MARCA
            nom = nom.replace("N sprint", "Clasif. Sprint").replace("N al sprint", "Clasif. Sprint")
            nom = nom.replace("N carrera", "Clasificación").replace("N", "Clasificación")

            if any(p in nom.lower() for p in ['libres', 'clasifica', 'carrera', 'sprint']):
                clean_s.append(f"{nom}: {s_hora}")

        if clean_s:
            horarios_completos.append(f"- {nombre_dia} {num_dia} = " + " | ".join(clean_s))

    row['Horarios'] = ' \n\n '.join(horarios_completos) if horarios_completos else 'N/A'
    # ============================================================================================

    data.append(row)

df = pd.DataFrame(data)
df = df[df['GP'].str.contains('GP', na=False)].drop_duplicates(subset=['GP'])

print("✅ Tabla generada con tu base original y el bloque del sábado exacto:")
display(df)

In [ ]:

# ==================== EXTRACCIÓN DE ESPN (CORREGIDA) ====================
urls = [
    "https://espndeportes.espn.com/deporte-motor/f1/posiciones/_/grupo/construtores",
    "https://espndeportes.espn.com/deporte-motor/f1/posiciones"
]

tablas = {}

for i, url in enumerate(urls):
   # print(f"Extrayendo tablas de: {url}")
    try:
        # Extrae TODAS las tablas HTML de la página
        dfs = pd.read_html(url)

        # 1. ESPN divide todo en dos: Nombres (dfs[0]) y Puntos (dfs[1])
        df_nombres = dfs[0].dropna(how='all').reset_index(drop=True)
        df_puntos = dfs[1].dropna(how='all').reset_index(drop=True)

        # 2. Unimos (pegamos) las dos tablas una al lado de la otra
        df_completo = pd.concat([df_nombres, df_puntos], axis=1)

        # 3. Limpiamos la columna de Nombres (suele ser la primera, índice 0)
        col_nombre = df_completo.columns[0]

        def limpiar_espn(texto):
            texto = str(texto)
            # Quitamos los números de posición del inicio (ej. "1MercedesMercedes" -> "MercedesMercedes")
            texto = re.sub(r'^\d+', '', texto).strip()

            # Cortamos a la mitad. Si las dos mitades son iguales, es un duplicado de ESPN
            mitad = len(texto) // 2
            if texto[:mitad] == texto[mitad:]:
                return texto[:mitad] # Devolvemos solo la primera mitad ("Mercedes")

            return texto

        # Aplicamos la limpieza a toda la columna
        df_completo[col_nombre] = df_completo[col_nombre].apply(limpiar_espn)

        # Le ponemos un título bonito dependiendo de la URL
        if "construtores" in url:
            df_completo.rename(columns={col_nombre: 'Escudería'}, inplace=True)
        else:
            df_completo.rename(columns={col_nombre: 'Piloto'}, inplace=True)

        # Guardamos la tabla limpia
        tablas[f"tabla_{i+1}"] = df_completo
        #print("✔️ Tabla unida y limpiada con éxito:")
        #display(df_completo.head())

    except Exception as e:
        print(f"Error en {url}: {e}")

# Bautizamos nuestras 3 tablas para no confundirnos
df_calendario = df
df_constructores = tablas.get("tabla_1", pd.DataFrame())
df_pilotos = tablas.get("tabla_2", pd.DataFrame())

# ==================== SUBIDA A GOOGLE SHEETS (MÚLTIPLES PESTAÑAS) ====================
from gspread_dataframe import set_with_dataframe
from gspread.exceptions import SpreadsheetNotFound, WorksheetNotFound

nombre_archivo = "F1_Base_De_Datos_2026"
CARPETA_ID = "1L1rrEY7evpCdmaHh0-gOYh7nPdtys3tW"

# 1. Buscamos o creamos el libro (el archivo principal)
try:
    sh = gc.open(nombre_archivo)
    print(f"\n El archivo '{nombre_archivo}' ya existe.")
except SpreadsheetNotFound:
    sh = gc.create(nombre_archivo, folder_id=CARPETA_ID)
    print(f"\nEl archivo '{nombre_archivo}' fue creado desde cero.")

# 2. Creamos una herramienta para guardar tablas en pestañas distintas
def guardar_en_hoja(libro_id, nombre_pestaña, dataframe):
    if dataframe.empty:
        print(f"La tabla para '{nombre_pestaña}' está vacía. Saltando...")
        
        return
    # RE-ABRIMOS el libro por ID para asegurar que tenemos la versión más reciente de Drive
    libro = gc.open_by_key(libro_id)

    try:
        # Intenta abrir la pestaña si ya existe
        hoja = libro.worksheet(nombre_pestaña)
        hoja.clear() # La limpia antes de pegar lo nuevo
    except WorksheetNotFound:
        # Si la pestaña no existe, la crea
        hoja = libro.add_worksheet(title=nombre_pestaña, rows="100", cols="30")

    # Pega los datos
    set_with_dataframe(hoja, dataframe, index=False)
    print(f"Tabla guardada en la pestaña: '{nombre_pestaña}'")
    
id_archivo = sh.id

# 3. Guardamos cada tabla en su propia pestaña
guardar_en_hoja(sh, "Calendario_MARCA", df_calendario)
guardar_en_hoja(sh, "Constructores_ESPN", df_constructores)
guardar_en_hoja(sh, "Pilotos_ESPN", df_pilotos)

print("\n¡Datos subidos correctamente! Puedes ver tu libro aquí:")
print(sh.url)
# ====================================================================================